In [1]:
import os
import tarfile
from pathlib import Path

import polars as pl
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
from polars import selectors as cs


In [2]:
raw_path = Path(os.environ["DATA_DIR"], "raw", "mbd_mini")
raw_path.mkdir(exist_ok=True)
snapshot_download(
    repo_id="ai-lab/MBD-mini",
    repo_type="dataset",
    local_dir=raw_path,
)


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

'/mnt/data/raw/mbd_mini'

In [3]:
if not (raw_path / "detail").exists():
    with tarfile.open(raw_path / "detail.tar.gz", "r:gz") as tar:
        tar.extractall(raw_path)

In [4]:
if not (raw_path / "targets").exists():
    with tarfile.open(raw_path / "targets.tar.gz", "r:gz") as tar:
        tar.extractall(raw_path)


In [5]:
trx = pl.read_parquet(raw_path / "detail" / "trx" / "fold=0").drop_nulls()
target = pl.read_parquet(raw_path / "targets" / "fold=0").drop_nulls()

In [6]:
target_mon = target.with_columns(
    pl.col("^target_.$").cast(pl.Boolean),
    mon=pl.col("mon").str.split("-").list.slice(0, 2).list.join("-"),
).drop("trans_count", "diff_trans_date")
trx_mon = trx.with_columns(mon=pl.col("event_time").dt.strftime("%Y-%m"))
trx_targ = trx_mon.join(target_mon, on=["client_id", "mon"])

In [7]:
trx_filt = trx_targ.filter(
    (pl.col("event_time").max() - pl.col("event_time").min()).over("client_id", "mon")
    > pl.duration(weeks=1),
    pl.len().over("client_id", "mon").is_between(32, 96),
    pl.col("currency") == 11,
).drop("src_type31", "src_type21", "currency")

In [8]:
trx_proc = trx_filt.with_columns(
    cs.integer().rank("dense").cast(pl.Int32),
    amount=pl.col("amount").abs().log1p() * pl.col("amount").sign(),
    time=(pl.col("event_time") - pl.col("event_time").min()).over("client_id", "mon")
    / (pl.duration(weeks=4)).over("client_id", "mon").median(),
).drop("event_time")

In [9]:
df = (
    trx_proc.sort("client_id", "time")
    .group_by("client_id", "mon")
    .agg(pl.exclude("^target_.$"), pl.first("^target_.$"))
).drop("client_id", "mon")

df_targ = df.with_columns(
    target=pl.concat_list([f"target_{i}" for i in range(1, 5)])
).drop("^target_.$")

In [10]:
train_val, test = train_test_split(df_targ, test_size=0.2)
train, val = train_test_split(train_val, test_size=0.2)

train = train.with_columns(split=pl.lit("train"))
val = val.with_columns(split=pl.lit("val"))
test = test.with_columns(split=pl.lit("test"))
df_final = pl.concat([train, val, test])

In [11]:
df_final.select(pl.col(pl.List(pl.Int32)).list.explode().n_unique()).row(0, named=True)

{'event_type': 52,
 'event_subtype': 56,
 'src_type11': 32,
 'src_type12': 108,
 'dst_type11': 40,
 'dst_type12': 150,
 'src_type22': 82,
 'src_type32': 81}

In [12]:
df_final.select(pl.col(pl.List(pl.Int32)).list.explode().max()).row(0, named=True)

{'event_type': 52,
 'event_subtype': 56,
 'src_type11': 32,
 'src_type12': 108,
 'dst_type11': 40,
 'dst_type12': 150,
 'src_type22': 82,
 'src_type32': 81}

In [13]:
df_final.select(pl.col("split").eq("train").sum())

split
u32
20950


In [14]:
df_final.write_parquet(
    Path(os.environ["DATA_DIR"], "preprocessed", "mbd_micro.parquet")
)